## Warehouse Manager Agent

In [1]:
from qdrant_client import QdrantClient
import random
import numpy as np
import psycopg2
from psycopg2.extras import RealDictCursor,execute_batch

c:\Users\jaysi\Desktop\Desktop\Ai-engineering\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
qdrant_client=QdrantClient(url="http://localhost:6333")

c:\Users\jaysi\Desktop\Desktop\Ai-engineering\.venv\Lib\site-packages\qdrant_client\qdrant_remote.py:290: UserWarning: Failed to obtain server version. Unable to check client-server compatibility. Set check_compatibility=False to skip version check.
  show_warning(


Fictional WareHouses

In [3]:
warehouses = [
    {
        "warehouse_id": "DE-BER-01",
        "warehouse_location": "Berlin, Germany",
        "warehouse_name": "Berlin Distribution Center"
    },
    {
        "warehouse_id": "DE-MUN-01",
        "warehouse_location": "Munich, Germany",
        "warehouse_name": "Munich Logistics Hub"
    },
    {
        "warehouse_id": "DE-HAM-01",
        "warehouse_location": "Hamburg, Germany",
        "warehouse_name": "Hamburg North Warehouse"
    },
    {
        "warehouse_id": "FR-PAR-01",
        "warehouse_location": "Paris, France",
        "warehouse_name": "Paris Central Depot"
    },
    {
        "warehouse_id": "FR-LYO-01",
        "warehouse_location": "Lyon, France",
        "warehouse_name": "Lyon Regional Warehouse"
    },
    {
        "warehouse_id": "FR-MAR-01",
        "warehouse_location": "Marseille, France",
        "warehouse_name": "Marseille Mediterranean Hub"
    }
]

### Simulate Stock Availibility For Each of the WareHouse

Retrieve all items IDs from Amazon items Qdrant Collection

In [4]:
import numpy as np
dummy_vector=np.zeros(1536).tolist()

In [5]:
payload=qdrant_client.query_points(
    collection_name="Amazon-items-collection-02-hybrid-serach",
    query=dummy_vector,
    using="text-embedding-model-3-small",
    limit=1000,
    with_payload=['parent_asin'],
    with_vectors=False
)

In [9]:
payload.points

[ScoredPoint(id=453, version=1, score=0.0, payload={'parent_asin': 'B0BYYGZHG5'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=404, version=1, score=0.0, payload={'parent_asin': 'B09VNTZBPG'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=284, version=1, score=0.0, payload={'parent_asin': 'B09QG1CC6H'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=132, version=1, score=0.0, payload={'parent_asin': 'B0B11P5XHV'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=94, version=1, score=0.0, payload={'parent_asin': 'B09TX32DMD'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=251, version=1, score=0.0, payload={'parent_asin': 'B0B97GBHXB'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=240, version=1, score=0.0, payload={'parent_asin': 'B0C3XS6N81'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=102, version=1, score=0.0, payload={'parent_asin': 'B09PYFMTBF'}, vector=No

In [14]:
parent_asins=[points.payload['parent_asin'] for points in payload.points]

In [15]:
parent_asins

['B0BYYGZHG5',
 'B09VNTZBPG',
 'B09QG1CC6H',
 'B0B11P5XHV',
 'B09TX32DMD',
 'B0B97GBHXB',
 'B0C3XS6N81',
 'B09PYFMTBF',
 'B0BN54T6XD',
 'B0C6H8B5TZ',
 'B09RKH6ST3',
 'B09L9MXVPL',
 'B09NVVGGJ6',
 'B0BF983MDW',
 'B09NQ16KJ9',
 'B0BXCF92XH',
 'B09VGH2LJC',
 'B0BTDJ6FBR',
 'B0B8S4D469',
 'B0B2NVGBN2',
 'B09Q8FLNDR',
 'B09WCL37Z4',
 'B0B62LR2YT',
 'B09R4JB5Q3',
 'B0CCW5D1NQ',
 'B09X2W94RP',
 'B0BH3X1HGH',
 'B0C1NP5KYD',
 'B0C7492QYK',
 'B0BNPVBZPG',
 'B0BHF59SHH',
 'B0CFG8BS2G',
 'B0BZSGNYXM',
 'B0BF9DQB6K',
 'B0BFPZGYLD',
 'B0B3J7V1V6',
 'B0B67M9C9P',
 'B09PZ8W2ZG',
 'B0BKF24BD5',
 'B09NMR7KMG',
 'B0CBPP4B74',
 'B0B6894XGM',
 'B0C4P5X7XB',
 'B0BR18QCCF',
 'B0C65TKK8V',
 'B09VDNKL4G',
 'B0CFY4JYFZ',
 'B09XXGDTPY',
 'B09Y77N1T2',
 'B0BGR94M7C',
 'B0BFBJFKFB',
 'B09GNFDH9M',
 'B09Y5WB2MP',
 'B0C378ZL2K',
 'B07TF5QD6B',
 'B09YTNFYJF',
 'B09MRM2NGG',
 'B0BB2KFPBJ',
 'B0C7K8XHQF',
 'B0CCRP8LRQ',
 'B0BG54HV9X',
 'B0C7KJJ46S',
 'B09WCFC5D9',
 'B0BCPJVSFM',
 'B0BGRL2618',
 'B09R1WB2S5',
 'B09YB3ZM

### Generate Synthatic Availability for all items in Qdrant

In [16]:
def generate_inventory_data(warehouses, product_ids, availability_rate=0.75):
    
    inventory_records = []
    
    for warehouse in warehouses:
        for product_id in product_ids:
            # 75% chance the product is available in this warehouse
            if random.random() < availability_rate:
                total_quantity = random.randint(0, 100)
                
                # Only add to inventory if quantity > 0
                if total_quantity > 0:
                    inventory_records.append({
                        "warehouse_id": warehouse["warehouse_id"],
                        "warehouse_location": warehouse["warehouse_location"],
                        "warehouse_name": warehouse["warehouse_name"],
                        "product_id": product_id,
                        "total_quantity": total_quantity,
                        "reserved_quantity": 0  # Starting with no reservations
                    })
    
    return inventory_records

In [17]:
stock_data = generate_inventory_data(warehouses, parent_asins)

In [19]:
stock_data

[{'warehouse_id': 'DE-BER-01',
  'warehouse_location': 'Berlin, Germany',
  'warehouse_name': 'Berlin Distribution Center',
  'product_id': 'B0BYYGZHG5',
  'total_quantity': 77,
  'reserved_quantity': 0},
 {'warehouse_id': 'DE-BER-01',
  'warehouse_location': 'Berlin, Germany',
  'warehouse_name': 'Berlin Distribution Center',
  'product_id': 'B0B11P5XHV',
  'total_quantity': 77,
  'reserved_quantity': 0},
 {'warehouse_id': 'DE-BER-01',
  'warehouse_location': 'Berlin, Germany',
  'warehouse_name': 'Berlin Distribution Center',
  'product_id': 'B09TX32DMD',
  'total_quantity': 83,
  'reserved_quantity': 0},
 {'warehouse_id': 'DE-BER-01',
  'warehouse_location': 'Berlin, Germany',
  'warehouse_name': 'Berlin Distribution Center',
  'product_id': 'B09PYFMTBF',
  'total_quantity': 54,
  'reserved_quantity': 0},
 {'warehouse_id': 'DE-BER-01',
  'warehouse_location': 'Berlin, Germany',
  'warehouse_name': 'Berlin Distribution Center',
  'product_id': 'B0BN54T6XD',
  'total_quantity': 76,
  

### Write Syntehetic Data To Postgres

In [20]:
from dotenv import load_dotenv
import os
load_dotenv()

True

In [27]:
def insert_inventory_to_db(inventory_records):

    try:
        # Connect to the database
        conn = psycopg2.connect(
            host="localhost",
            port=os.getenv("DB_PORT"),
            database=os.getenv("DB_NAME"),
            user=os.getenv("DB_USER"),
            password=os.getenv("DB_PASSWORD")
        )
        conn.autocommit = True

        with conn.cursor(cursor_factory=RealDictCursor) as cursor:

            # Prepare the INSERT query
            insert_query = """
            INSERT INTO warehouses.inventory 
            (warehouse_id, warehouse_location, warehouse_name, product_id, total_quantity, reserved_quantity)
            VALUES (%(warehouse_id)s, %(warehouse_location)s, %(warehouse_name)s, %(product_id)s, %(total_quantity)s, %(reserved_quantity)s)
            """
            
            # Use execute_batch for better performance with many inserts
            execute_batch(cursor, insert_query, inventory_records, page_size=100)
            
            # Commit the transaction
            conn.commit()
            
            print(f"Successfully inserted {len(inventory_records)} records into warehouses.inventory")
            
            # Close cursor and connection
            cursor.close()
            conn.close()
        
    except psycopg2.Error as e:
        print(f"Database error: {e}")
        if conn:
            conn.rollback()
    except Exception as e:
        print(f"Error: {e}")
    finally:
        if cursor:
            cursor.close()
        if conn:
            conn.close()

In [28]:
insert_inventory_to_db(stock_data)

Successfully inserted 4449 records into warehouses.inventory
